# 01. 데이터 전처리

**제14회 2026 빅콘테스트 AI데이터 분석 분야 — 기후 위기에 따른 취약 상권 영향 분석**

| 항목 | 내용 |
|---|---|
| 목적 | 대회 제공 데이터(신한카드·SKT 유동인구)를 점검·정제하고, 이후 단계(기술통계·EDA·피해 예측·지원금 최적화)가 공통으로 쓰는 분석용 테이블을 만든다 |
| 입력 | 대회 데이터: `data/신한카드_데이터/` (데이터1·2), `data/sk통신인구_데이터/` (성연령·시간대·요일 × 6개월)<br>외부 공공데이터: 기상청 ASOS 시간·일자료, 소상공인 상가(상권)정보<br>기준표: `references/` (업종 매핑표, 상가 업종 매핑표, 공휴일) |
| 출력 | `data/processed/*.parquet` (git 제외) |
| 브랜치 | `ana-prep` |

**진행 순서**
1. 환경 설정
2. 카드 데이터 로딩·스키마 점검
3. 카드 데이터 정제 (지역·성별·연령·법인 구분, 업종 매핑, 달력 변수)
4. 데이터1 ↔ 데이터2 정합성 점검
5. 분석 마트 생성 (균형 패널)
6. SKT 유동인구 로딩·정제·프로파일
7. 기상 데이터 (기상청 ASOS) — 체감온도, 기상 유형, 평년값
8. 소상공인 상가정보 — 업종별 점포 수·체인 비중
9. 분석 테이블 생성 (카드 × 기상 결합)
10. 검증 요약·저장

> 대회 원본은 외부 공유가 금지되어 있으므로, 이 노트북은 원본 행을 그대로 출력하지 않고 **집계·요약 결과만** 보여준다.

## 1. 환경 설정

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
CARD_DIR = DATA / '신한카드_데이터'
FLOW_DIR = DATA / 'sk통신인구_데이터'
REF = ROOT / 'references'
OUT = DATA / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

CARD_FILE_1 = CARD_DIR / '신한카드_빅콘테스트2026_데이터1.txt'       # 일별 × 6시간대 × 업종 × 성·연령 (법인 포함)
CARD_FILE_2 = CARD_DIR / '신한카드_빅콘테스트2026_데이터2_수정.txt'  # 일별 × 업종 × 고객 거주 시도 × 성·연령 (법인 제외)

PERIOD = pd.date_range('2025-07-01', '2025-12-31', name='date')
REGIONS = {  # 카드 가맹점 지역 → 분석 지역명, 고객 '현지' 판정용 시도, SKT 시군구 코드(BLOCK_CD 앞 5자리)
    '서울 강남구': {'region': '강남구', 'home_sido': '서울', 'flow_sgg': '11230'},
    '강원 춘천시': {'region': '춘천시', 'home_sido': '강원', 'flow_sgg': '32010'},
}
TIME_SLOTS = ['00_05', '06_11', '12_17', '18_23']
WEEKDAYS = ['월', '화', '수', '목', '금', '토', '일']
print('ROOT =', ROOT)
for p in [CARD_FILE_1, CARD_FILE_2, FLOW_DIR, REF / 'industry_map.csv', REF / 'holidays_2025H2.csv']:
    print(f"{'OK ' if p.exists() else '없음'} {p.relative_to(ROOT)}")

ROOT = C:\Users\CoBa\bigcontest_2026\eda_analysis
OK  data\신한카드_데이터\신한카드_빅콘테스트2026_데이터1.txt
OK  data\신한카드_데이터\신한카드_빅콘테스트2026_데이터2_수정.txt
OK  data\sk통신인구_데이터
OK  references\industry_map.csv
OK  references\holidays_2025H2.csv


## 2. 카드 데이터 로딩·스키마 점검

- 두 파일 모두 **cp949 인코딩, 탭 구분**이다.
- 모든 컬럼을 문자열로 읽은 뒤 직접 변환한다 (코드값 앞자리 0 손실, 자동 형변환 오류 방지).

In [2]:
COLS_1 = ['TA_YMD', 'TIME_GB', 'MCT_SGG_CD', 'MCT_RY_CD', 'SEX_CCD', 'AGE_CCD', 'TS_AT', 'USE_CNT']
COLS_2 = ['TA_YMD', 'MCT_SGG_CD', 'MCT_RY_CD', 'CLN_SGG_CD', 'SEX_CCD', 'AGE_CCD', 'TS_AT', 'USE_CNT']

def load_card(path, expected_cols):
    raw = pd.read_csv(path, sep='\t', encoding='cp949', dtype=str, keep_default_na=False)
    assert list(raw.columns) == expected_cols, f'예상과 다른 컬럼: {list(raw.columns)}'
    return raw.apply(lambda s: s.str.strip())

raw1 = load_card(CARD_FILE_1, COLS_1)
raw2 = load_card(CARD_FILE_2, COLS_2)
print(f'데이터1: {raw1.shape[0]:,}행 × {raw1.shape[1]}열')
print(f'데이터2: {raw2.shape[0]:,}행 × {raw2.shape[1]}열')

데이터1: 1,044,710행 × 8열
데이터2: 1,806,800행 × 8열


### 2-1. 컬럼별 값 점검 (결측·고유값 수·값 범위)

In [3]:
def schema_report(raw):
    rows = []
    for c in raw.columns:
        s = raw[c]
        rows.append({'컬럼': c, '빈 문자열': int((s == '').sum()), '고유값 수': s.nunique(),
                     '예시(상위 빈도)': ', '.join(s.value_counts().head(4).index)})
    return pd.DataFrame(rows)

display(schema_report(raw1))
display(schema_report(raw2))

,컬럼,빈 문자열,고유값 수,예시(상위 빈도)
0,TA_YMD,0,184,"20250801, 20250731, 20250926, 20251002"
1,TIME_GB,0,4,"12_17, 06_11, 18_23, 00_05"
2,MCT_SGG_CD,0,2,"서울 강남구, 강원 춘천시"
3,MCT_RY_CD,0,91,"편의점, 한식, 기타요식, ZZ_나머지"
4,SEX_CCD,0,3,"남성, 여성, 법인"
5,AGE_CCD,0,10,"40 대, 30 대, 50 대, 20 대"
6,TS_AT,0,732376,"54697, 54798, 53948, 54462"
7,USE_CNT,0,19494,"5, 6, 11, 12"


,컬럼,빈 문자열,고유값 수,예시(상위 빈도)
0,TA_YMD,0,184,"20250801, 20250725, 20250731, 20251115"
1,MCT_SGG_CD,0,2,"서울 강남구, 강원 춘천시"
2,MCT_RY_CD,0,90,"할인점/슈퍼마켓/양판점, 편의점, 한식, ZZ_나머지"
3,CLN_SGG_CD,0,17,"서울, 경기, 강원, 인천"
4,SEX_CCD,0,2,"남성, 여성"
5,AGE_CCD,0,9,"30 대, 40 대, 50 대, 20 대"
6,TS_AT,0,978730,"54697, 53948, 53935, 54798"
7,USE_CNT,0,16499,"5, 6, 11, 12"


In [4]:
def key_checks(raw, name):
    keys = [c for c in raw.columns if c not in ('TS_AT', 'USE_CNT')]
    amt, cnt = raw['TS_AT'].astype('int64'), raw['USE_CNT'].astype('int64')
    return pd.Series({
        '기간': f"{raw['TA_YMD'].min()} ~ {raw['TA_YMD'].max()}",
        '일수': raw['TA_YMD'].nunique(),
        '완전 중복 행': int(raw.duplicated().sum()),
        '키 중복 행': int(raw.duplicated(keys).sum()),
        '결제금액 ≤ 0': int((amt <= 0).sum()),
        '결제건수 최솟값': int(cnt.min()),
        '결제금액 합계(조원)': round(amt.sum() / 1e12, 2),
    }, name=name)

pd.concat([key_checks(raw1, '데이터1'), key_checks(raw2, '데이터2')], axis=1)

,데이터1,데이터2
기간,20250701 ~ 20251231,20250701 ~ 20251231
일수,184,184
완전 중복 행,0,0
키 중복 행,0,0
결제금액 ≤ 0,0,0
결제건수 최솟값,5,5
결제금액 합계(조원),35.87,30.15


**점검 결과**
- 결측(빈 문자열)·중복 키·음수 금액은 없다.
- **결제건수 최솟값이 5**다. 결제 5건 미만인 칸은 비식별 조치로 원본에서 빠졌다는 뜻이다. 그래서 마트를 만들 때 비어 있는 칸은 '매출 0'이 아니라 **'소액이라 미집계'**를 포함한다고 해석한다.
- 데이터1에만 `법인`(성별·연령 자리에 '법인')이 있고, 데이터2는 법인 결제가 빠져 있다.
- 데이터2에만 `통신요금(PC통신,무선호출)` 업종이 없다(비소비성 업종이라 분석에서 제외).

## 3. 카드 데이터 정제

### 3-1. 업종 매핑표
업종 91개를 **기상에 비슷하게 반응할 것으로 보이는 17개 대분류**로 묶었다 (`references/industry_map.csv`).
- `analysis_include = N`: 세금·보험·통신요금·상품권·미분류처럼 **소비가 아닌 결제**, 등록금처럼 **정기 납부**, 수입차·부동산처럼 **고액·비정기**, B2B, 결제 위치와 소비 위치가 다른 업종은 분석에서 뺀다.
- 가설 3(기후별로 업종 피해가 다르다)을 보기 위해 **카페·디저트(커피·제과)**와 **간편식(패스트푸드)**은 음식점과 따로 둔다.

In [5]:
industry_map = pd.read_csv(REF / 'industry_map.csv', dtype=str, keep_default_na=False)
holidays = pd.read_csv(REF / 'holidays_2025H2.csv', parse_dates=['date'])

unmapped = set(raw1['MCT_RY_CD']) | set(raw2['MCT_RY_CD'])
unmapped -= set(industry_map['industry'])
assert not unmapped, f'매핑표에 없는 업종: {unmapped}'

summary = (industry_map.groupby(['industry_group', 'analysis_include']).size()
           .unstack(fill_value=0).rename(columns={'Y': '포함', 'N': '제외'}))
print(f"업종 {len(industry_map)}개 → 대분류 {industry_map['industry_group'].nunique()}개 "
      f"(분석 포함 {(industry_map['analysis_include'] == 'Y').sum()}개 업종)")
summary

업종 91개 → 대분류 17개 (분석 포함 71개 업종)


analysis_include,제외,포함
industry_group,,
간편식,0,1
교육,2,2
교통·자동차,2,8
기타소매,1,16
대형유통,0,4
미용·개인서비스,0,5
비소비성,7,0
생활서비스,6,0
숙박,1,2


### 3-2. 공통 정제 + 데이터별 정제
| 원본 | 정제 컬럼 | 처리 |
|---|---|---|
| TA_YMD | `date` | 날짜형 |
| MCT_SGG_CD | `region` | `강남구` / `춘천시` |
| MCT_RY_CD | `industry`, `industry_group`, `analysis_include` | 매핑표 결합 |
| SEX_CCD | `sex`, `customer_type` | `M`/`F`, 법인은 결측 + 고객유형 `법인` |
| AGE_CCD | `age` | `20 대` → 20 (법인은 결측) |
| TIME_GB | `time_slot` | 데이터1만, 6시간 구간 |
| CLN_SGG_CD | `customer_sido`, `customer_origin` | 데이터2만. 가맹점과 같은 시도면 `현지`, 다르면 `외지`, 정보없음은 `미상` |
| TS_AT, USE_CNT | `amount`, `count` | 정수형 |

In [6]:
ORIGINS = ['현지', '외지', '미상']

def add_calendar(df):
    hol = df['date'].map(holidays.set_index('date')['holiday_name'])
    dow = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['dow'] = pd.Categorical(dow.map(dict(enumerate(WEEKDAYS))), categories=WEEKDAYS, ordered=True)
    df['is_weekend'] = dow >= 5
    df['is_holiday'] = hol.notna()
    df['holiday_name'] = hol
    df['is_offday'] = df['is_weekend'] | df['is_holiday']   # 휴일 = 주말 또는 공휴일
    df['is_payday_week'] = df['date'].dt.day >= 25            # 월급일 전후(25일~말일) 소비 증가 통제용
    return df

def clean_common(raw):
    ind = industry_map.set_index('industry')
    df = pd.DataFrame({
        'date': pd.to_datetime(raw['TA_YMD'], format='%Y%m%d'),
        'region': raw['MCT_SGG_CD'].map({k: v['region'] for k, v in REGIONS.items()}),
        'industry': raw['MCT_RY_CD'],
        'sex': raw['SEX_CCD'].map({'남성': 'M', '여성': 'F'}),
        'age': pd.to_numeric(raw['AGE_CCD'].str.replace(' 대', '', regex=False), errors='coerce').astype('Int64'),
        'amount': raw['TS_AT'].astype('int64'),
        'count': raw['USE_CNT'].astype('int64'),
    })
    assert df['region'].notna().all(), '분석 지역 외 가맹점이 있음'
    df['industry_group'] = df['industry'].map(ind['industry_group'])
    df['analysis_include'] = df['industry'].map(ind['analysis_include']).eq('Y')
    return add_calendar(df)

card1 = clean_common(raw1)
card1.insert(1, 'time_slot', pd.Categorical(raw1['TIME_GB'], categories=TIME_SLOTS, ordered=True))
card1['customer_type'] = np.where(raw1['SEX_CCD'].eq('법인'), '법인', '개인')

card2 = clean_common(raw2)
sido = raw2['CLN_SGG_CD'].replace('정보없음', pd.NA)
home = raw2['MCT_SGG_CD'].map({k: v['home_sido'] for k, v in REGIONS.items()})
card2.insert(4, 'customer_sido', sido)
card2.insert(5, 'customer_origin', pd.Categorical(
    np.where(sido.isna(), '미상', np.where(sido == home, '현지', '외지')), categories=ORIGINS))

assert card1['time_slot'].notna().all()
print('card1', card1.shape, '| card2', card2.shape)
card1.dtypes.to_frame('dtype').T

card1 (1044710, 18) | card2 (1806800, 18)


,date,time_slot,region,industry,sex,age,amount,count,industry_group,analysis_include,month,dow,is_weekend,is_holiday,holiday_name,is_offday,is_payday_week,customer_type
dtype,datetime64[us],category,str,str,str,Int64,int64,int64,str,bool,int32,category,bool,bool,str,bool,bool,str


### 3-3. 분석에서 빠지는 금액은 얼마나 되나

In [7]:
total = card1['amount'].sum()
share = (card1.groupby(['customer_type', 'analysis_include'])['amount'].sum() / total * 100).unstack()
share.columns = ['분석 제외 업종', '분석 포함 업종']
print('데이터1 결제금액 비중(%) — 고객유형 × 업종 포함 여부')
display(share.round(1))

excl = (card1[~card1['analysis_include']].groupby('industry')['amount'].sum() / total * 100).sort_values(ascending=False)
print('제외 업종 상위 5 (전체 금액 대비 %)')
excl.head().round(1)

데이터1 결제금액 비중(%) — 고객유형 × 업종 포함 여부

,분석 제외 업종,분석 포함 업종
customer_type,,
개인,46.60,37.40
법인,9.80,6.10


제외 업종 상위 5 (전체 금액 대비 %)


industry
ZZ_나머지      26.10
세금공과금       24.60
보험           3.10
상품권/복권       1.30
예식장/결혼서비스    0.50
Name: amount, dtype: float64

- 세금·보험·미분류(`ZZ_나머지`)·법인 결제가 금액의 큰 몫을 차지한다. 이들은 날씨와 무관한 납부성·B2B 결제라 그대로 두면 기상 효과가 묻힌다.
- 이후 분석은 **개인 결제 × 분석 포함 업종**만 쓴다 (원본 행은 `analysis_include`, `customer_type` 플래그로 보존).

## 4. 데이터1 ↔ 데이터2 정합성 점검
데이터2(고객 거주지 기준)는 법인을 뺀 같은 결제를 다른 기준으로 나눈 것으로 보인다.
날짜 × 지역 × 업종 단위로 **데이터1 개인 결제 합계와 데이터2 합계**를 비교한다.

In [8]:
keys = ['date', 'region', 'industry']
a1 = card1[card1['customer_type'] == '개인'].groupby(keys)['amount'].sum()
a2 = card2.groupby(keys)['amount'].sum()
both = pd.concat({'card1_개인': a1, 'card2': a2}, axis=1).fillna(0)
both['상대오차'] = (both['card2'] - both['card1_개인']).abs() / both[['card1_개인', 'card2']].max(axis=1)

print(f"비교 칸 {len(both):,}개 | 완전 일치 {(both['상대오차'] == 0).mean():.1%} | "
      f"상대오차 1% 초과 {(both['상대오차'] > 0.01).mean():.2%} | 최대 {both['상대오차'].max():.2%}")
reg = both.groupby('region')[['card1_개인', 'card2']].sum()
reg['차이(%)'] = (reg['card2'] / reg['card1_개인'] - 1) * 100
(reg.assign(card1_개인=reg['card1_개인'] / 1e8, card2=reg['card2'] / 1e8)
    .rename(columns={'card1_개인': '데이터1 개인(억원)', 'card2': '데이터2(억원)'}).round(3))

비교 칸 30,067개 | 완전 일치 28.0% | 상대오차 1% 초과 0.00% | 최대 0.00%


,데이터1 개인(억원),데이터2(억원),차이(%)
region,,,
강남구,"280,899.43","280,899.43",-0.00
춘천시,"20,611.42","20,611.42",-0.00


- 두 데이터는 같은 개인 결제를 **시간대 기준(데이터1)과 고객 거주지 기준(데이터2)**으로 각각 나눈 것이다.
- 따라서 **기상 효과(시간대·성·연령)는 데이터1**, **현지·외지 고객 분석은 데이터2**로 하고, 두 결과의 총액이 같다고 보고 함께 해석할 수 있다.

## 5. 분석 마트 생성 (균형 패널)

원본은 결제 5건 미만 칸이 없어서, 그대로 집계하면 **어떤 날은 행 자체가 빠진** 시계열이 된다.
날씨 효과를 보려면 모든 날짜가 있어야 하므로 **날짜 × 지역 × 업종 전체 조합을 만들고 빈 칸을 0으로 채운다** (`n_cells = 0`이면 원본에 없던 칸).

| 마트 | 단위 | 원천 | 용도 |
|---|---|---|---|
| `mart_daily_group` | 날짜 × 지역 × 업종대분류 | 데이터1 개인 | 기상 효과·피해 예측의 기본 단위 |
| `mart_daily_industry` | 날짜 × 지역 × 세부 업종 | 데이터1 개인 | 가설 3 (같은 대분류 안에서도 업종별 차이) |
| `mart_slot_group` | 날짜 × 시간대 × 지역 × 업종대분류 | 데이터1 개인 | 시간대별 기상 효과 |
| `mart_origin_group` | 날짜 × 고객출신 × 지역 × 업종대분류 | 데이터2 | 현지·외지 고객 이탈 |
| `mart_daily_demo` | 날짜 × 지역 × 성 × 연령대 | 데이터1 개인 | 고객층별 반응 (분석 포함 업종 합계) |

In [9]:
CAL_COLS = ['month', 'dow', 'is_weekend', 'is_holiday', 'holiday_name', 'is_offday', 'is_payday_week']
calendar = add_calendar(pd.DataFrame({'date': PERIOD}))[['date'] + CAL_COLS]

def balanced_mart(base, keys, levels):
    """keys 단위 합계를 levels의 모든 조합으로 펼치고 빈 칸을 0으로 채운다 (날짜는 184일 전체)."""
    agg = (base.groupby(['date'] + keys, observed=True)
               .agg(amount=('amount', 'sum'), count=('count', 'sum'), n_cells=('amount', 'size')))
    idx = pd.MultiIndex.from_product([PERIOD] + [levels[k] for k in keys], names=['date'] + keys)
    out = agg.reindex(idx, fill_value=0).reset_index()
    return out.merge(calendar, on='date', how='left')

personal = card1[(card1['customer_type'] == '개인') & card1['analysis_include']]
target2 = card2[card2['analysis_include']]
regions = sorted(personal['region'].unique())
groups = sorted(personal['industry_group'].unique())
industries = sorted(personal['industry'].unique())

marts = {
    'mart_daily_group':    balanced_mart(personal, ['region', 'industry_group'],
                                         {'region': regions, 'industry_group': groups}),
    'mart_daily_industry': balanced_mart(personal, ['region', 'industry'],
                                         {'region': regions, 'industry': industries}),
    'mart_slot_group':     balanced_mart(personal, ['time_slot', 'region', 'industry_group'],
                                         {'time_slot': TIME_SLOTS, 'region': regions, 'industry_group': groups}),
    'mart_origin_group':   balanced_mart(target2, ['customer_origin', 'region', 'industry_group'],
                                         {'customer_origin': ORIGINS, 'region': regions, 'industry_group': groups}),
    'mart_daily_demo':     balanced_mart(personal, ['region', 'sex', 'age'],
                                         {'region': regions, 'sex': ['M', 'F'], 'age': sorted(personal['age'].dropna().unique())}),
}
# 세부 업종 마트에 대분류 붙이기
marts['mart_daily_industry'] = marts['mart_daily_industry'].merge(
    industry_map[['industry', 'industry_group']], on='industry', how='left')

pd.DataFrame({name: {'행 수': len(m), '0 채움 비율(%)': (m['n_cells'] == 0).mean() * 100,
                     '금액 합계(억원)': m['amount'].sum() / 1e8} for name, m in marts.items()}).T.round(1)

,행 수,0 채움 비율(%),금액 합계(억원)
mart_daily_group,"5,520.00",0.00,"134,227.40"
mart_daily_industry,"26,128.00",3.90,"134,227.40"
mart_slot_group,"22,080.00",2.70,"134,227.40"
mart_origin_group,"16,560.00",11.90,"134,227.40"
mart_daily_demo,"6,624.00",0.00,"134,227.40"


### 5-1. 0 채움이 많은 조합 (분석 시 주의 대상)

In [10]:
g = marts['mart_daily_group']
zero = (g.assign(zero=g['n_cells'] == 0).pivot_table(index='industry_group', columns='region', values='zero', aggfunc='mean') * 100)
print('업종대분류별 0 채움 일수 비율(%) — 일 단위')
display(zero.round(1).sort_values('춘천시', ascending=False))

ind = marts['mart_daily_industry']
zi = ind.groupby(['region', 'industry'])['n_cells'].apply(lambda s: (s == 0).mean() * 100)
print(f"세부 업종 × 지역 {len(zi)}개 중 0 채움 일수 20% 초과: {(zi > 20).sum()}개 "
      f"→ 세부 업종 분석은 0 채움이 적은 업종으로 한정한다")

업종대분류별 0 채움 일수 비율(%) — 일 단위


region,강남구,춘천시
industry_group,,
간편식,0.00,0.00
교육,0.00,0.00
교통·자동차,0.00,0.00
기타소매,0.00,0.00
대형유통,0.00,0.00
미용·개인서비스,0.00,0.00
숙박,0.00,0.00
식료품소매,0.00,0.00
여가·오락,0.00,0.00


세부 업종 × 지역 142개 중 0 채움 일수 20% 초과: 5개 → 세부 업종 분석은 0 채움이 적은 업종으로 한정한다


- 일 단위 업종대분류 마트는 0 채움이 거의 없어 그대로 시계열로 쓸 수 있다.
- 시간대(특히 `00_05`)·세부 업종으로 쪼개면 0이 늘어난다. 이 칸들은 '매출 없음'보다 '소액 미집계'일 가능성이 커서 **기술통계·EDA에서 표본 기준을 따로 정한다.**

## 6. SKT 유동인구 로딩·정제

### 원본 특성
- 50m 격자(`X_COORD`, `Y_COORD`, UTM-K) 단위의 **월별 일평균 유동인구**다. 특정 날짜 값이 아니므로 날씨와 일 단위로 연결할 수 없고, **상권 구조(누가·언제·평일/주말)를 설명하는 변수**로 쓴다.
- 파일 3종: 성·연령(12개 값), 시간대(24개), 요일(7개) × 6개월(2025년 7~12월). `|` 구분, UTF-8(BOM).

In [11]:
KEY = ['STD_YM', 'BLOCK_CD', 'X_COORD', 'Y_COORD']
AGES = ['10G', '20G', '30G', '40G', '50G', '60GU']
AGE_COLS = [f'{s}_FLOW_POP_CNT_{a}' for s in ['MAN', 'WMAN'] for a in AGES]
HOUR_COLS = [f'TMST_{h:02d}' for h in range(24)]
WKDY_COLS = ['FLOW_POP_CNT_MON', 'FLOW_POP_CNT_TUS', 'FLOW_POP_CNT_WED', 'FLOW_POP_CNT_THU',
             'FLOW_POP_CNT_FRI', 'FLOW_POP_CNT_SAT', 'FLOW_POP_CNT_SUN']
VALUE_COLS = {'age': AGE_COLS, 'time': HOUR_COLS, 'wkdy': WKDY_COLS}
MONTHS = ['202507', '202508', '202509', '202510', '202511', '202512']

flow_raw, load_log = {}, []
for kind, cols in VALUE_COLS.items():
    frames = []
    for ym in MONTHS:
        df = pd.read_csv(FLOW_DIR / f'flow_{kind}_pop_{ym}.csv', sep='|', encoding='utf-8-sig',
                         dtype={'STD_YM': str, 'BLOCK_CD': str})
        assert list(df.columns) == KEY + cols, f'{kind} {ym} 컬럼 불일치'
        dedup = df.drop_duplicates()
        load_log.append({'종류': kind, '월': ym, '원본 행': len(df), '완전 중복': len(df) - len(dedup),
                         '중복 제거 후 키 중복': int(dedup.duplicated(KEY).sum()),
                         '음수 값': int((dedup[cols] < 0).sum().sum())})
        frames.append(dedup)
    flow_raw[kind] = pd.concat(frames, ignore_index=True)

load_log = pd.DataFrame(load_log)
load_log.pivot(index='월', columns='종류', values=['원본 행', '완전 중복'])

원본 행                 완전 중복               
종류         age    time    wkdy   age   time    wkdy
월                                                  
202507   98129   94099  117825     0      0       0
202508   99115   95374  118958     0      0       0
202509   99469   95084  119865     0      0       0
202510  100750   97881  120643     0      0       0
202511  100444   95822  122248     0      0       0
202512   98345  185090  242534     0  92545  121267

- **12월 시간대·요일 파일은 모든 행이 두 번 들어 있다** (원본 행 수가 다른 달의 약 2배, 완전 중복 = 절반). 완전 중복을 제거하면 키 중복이 남지 않으므로 같은 행이 단순 반복된 것이다.

In [12]:
assert (load_log['중복 제거 후 키 중복'] == 0).all() and (load_log['음수 값'] == 0).all()

grid = (flow_raw['age'].merge(flow_raw['time'], on=KEY, how='outer')
                       .merge(flow_raw['wkdy'], on=KEY, how='outer'))
for kind, cols in VALUE_COLS.items():
    grid[f'has_{kind}'] = grid[cols[0]].notna()
all_vals = AGE_COLS + HOUR_COLS + WKDY_COLS
grid[all_vals] = grid[all_vals].fillna(0)

sgg = grid['BLOCK_CD'].str[:5]
sgg_to_region = {v['flow_sgg']: v['region'] for v in REGIONS.values()}
outside = ~sgg.isin(sgg_to_region)
print(f'격자×월 {len(grid):,}행 | 분석 지역 밖(인접 시군구 코드) {outside.sum():,}행 제외: '
      f"{sorted(sgg[outside].unique())}")
print('세 파일 중 일부에만 있는 격자×월 비율:',
      f"{(~grid[['has_age', 'has_time', 'has_wkdy']].all(axis=1)).mean():.1%} (없는 쪽 값은 0으로 채움)")

flow = grid[~outside].copy()
flow.insert(0, 'region', sgg[~outside].map(sgg_to_region))
flow.insert(1, 'month', flow['STD_YM'].str[4:].astype(int))
flow.insert(2, 'dong_cd', flow['BLOCK_CD'].str[:7])   # 앞 7자리 = 읍면동 단위 코드
flow.insert(3, 'grid_id', flow['X_COORD'].round().astype(int).astype(str) + '_' + flow['Y_COORD'].round().astype(int).astype(str))
assert not flow.duplicated(['month', 'grid_id']).any()
flow.groupby('region').agg(격자수=('grid_id', 'nunique'), 읍면동수=('dong_cd', 'nunique'), 행수=('grid_id', 'size'))

격자×월 721,318행 | 분석 지역 밖(인접 시군구 코드) 641행 제외: ['11030', '11040', '11220', '11240', '31370', '32310', '32370', '32380']
세 파일 중 일부에만 있는 격자×월 비율: 22.0% (없는 쪽 값은 0으로 채움)


,격자수,읍면동수,행수
region,,,
강남구,15541,22,92750
춘천시,110750,25,627927


### 6-1. 세 파일의 척도 일치 확인
성연령 합, 24시간 합, 요일 평균이 모두 '하루 평균 유동인구'라면 지역·월 단위 합계가 비슷해야 한다.

In [13]:
by = flow.groupby(['region', 'month'])
scale = pd.DataFrame({'성연령 합': by[AGE_COLS].sum().sum(axis=1),
                      '시간대 합': by[HOUR_COLS].sum().sum(axis=1),
                      '요일 평균': by[WKDY_COLS].sum().sum(axis=1) / 7})
scale['시간대/성연령'] = scale['시간대 합'] / scale['성연령 합']
scale['요일/성연령'] = scale['요일 평균'] / scale['성연령 합']
display((scale.assign(**{c: scale[c] / 1e3 for c in ['성연령 합', '시간대 합', '요일 평균']})).round(3))
assert ((scale['시간대/성연령'] - 1).abs() < 0.01).all() and ((scale['요일/성연령'] - 1).abs() < 0.03).all()
print('세 파일의 척도가 1~3% 안에서 일치 → 하나의 일평균 유동인구로 결합 가능')

성연령 합    시간대 합    요일 평균  시간대/성연령  요일/성연령
region month                                            
강남구    7     7,542.99 7,543.18 7,476.07     1.00    0.99
       8     7,184.34 7,184.53 7,280.38     1.00    1.01
       9     7,599.42 7,599.63 7,565.91     1.00    1.00
       10    7,132.36 7,132.67 7,057.80     1.00    0.99
       11    7,459.15 7,459.36 7,562.02     1.00    1.01
       12    7,320.64 7,320.85 7,259.71     1.00    0.99
춘천시    7     1,796.43 1,799.43 1,799.30     1.00    1.00
       8     1,860.69 1,863.77 1,864.53     1.00    1.00
       9     1,954.77 1,957.72 1,955.33     1.00    1.00
       10    1,889.35 1,892.48 1,889.32     1.00    1.00
       11    1,861.33 1,864.19 1,869.41     1.00    1.00
       12    1,774.36 1,777.05 1,773.59     1.00    1.00

세 파일의 척도가 1~3% 안에서 일치 → 하나의 일평균 유동인구로 결합 가능


### 6-2. 파생 변수와 프로파일
| 변수 | 정의 |
|---|---|
| `pop_total` | 일평균 유동인구 (성연령 합) |
| `share_age_10` ~ `share_age_60` | 연령대 비중 (60 = 60대 이상, 남녀 합) |
| `share_male` | 남성 비중 |
| `pop_slot_*`, `share_slot_*` | 카드 데이터와 같은 6시간 구간의 유동인구·비중 |
| `pop_weekday`, `pop_weekend`, `weekend_ratio` | 평일·주말 하루 평균, 주말 ÷ 평일 (1 미만이면 평일 중심 상권) |

비중은 집계 단위에서 **원값을 먼저 합한 뒤** 계산한다 (격자 비중의 단순평균이 아닌 인구 가중).

In [14]:
SLOT_HOURS = {'00_05': range(0, 6), '06_11': range(6, 12), '12_17': range(12, 18), '18_23': range(18, 24)}

def add_flow_features(df):
    total = df[AGE_COLS].sum(axis=1)
    safe = total.where(total > 0)
    df['pop_total'] = total
    for a in AGES:
        df[f'share_age_{a[:2]}'] = (df[f'MAN_FLOW_POP_CNT_{a}'] + df[f'WMAN_FLOW_POP_CNT_{a}']) / safe
    df['share_male'] = df[[c for c in AGE_COLS if c.startswith('MAN')]].sum(axis=1) / safe
    hour_total = df[HOUR_COLS].sum(axis=1).where(lambda s: s > 0)
    for slot, hours in SLOT_HOURS.items():
        df[f'pop_slot_{slot}'] = df[[f'TMST_{h:02d}' for h in hours]].sum(axis=1)
        df[f'share_slot_{slot}'] = df[f'pop_slot_{slot}'] / hour_total
    df['pop_weekday'] = df[WKDY_COLS[:5]].mean(axis=1)
    df['pop_weekend'] = df[WKDY_COLS[5:]].mean(axis=1)
    df['weekend_ratio'] = df['pop_weekend'] / df['pop_weekday'].where(df['pop_weekday'] > 0)
    return df

def flow_profile(df, keys, n_months=1):
    agg = df.groupby(keys)[all_vals].sum() / n_months
    agg['n_grids'] = df.groupby(keys)['grid_id'].nunique()
    return add_flow_features(agg.reset_index()).drop(columns=all_vals)

flow = add_flow_features(flow)
flow_region_month = flow_profile(flow, ['region', 'month'])
flow_dong = flow_profile(flow, ['region', 'dong_cd'], n_months=len(MONTHS))   # 6개월 평균
flow_grid = flow_profile(flow, ['region', 'dong_cd', 'grid_id', 'X_COORD', 'Y_COORD'], n_months=len(MONTHS))

cols = ['pop_total', 'share_age_20', 'share_age_60', 'share_slot_18_23', 'weekend_ratio']
view = flow_region_month.set_index(['region', 'month'])[cols].copy()
view['pop_total'] = view['pop_total'] / 1e3
view.rename(columns={'pop_total': '유동인구(천명)'}).round(3)

유동인구(천명)  share_age_20  share_age_60  share_slot_18_23  weekend_ratio
region month                                                                       
강남구    7      7,542.99          0.16          0.18              0.26           0.71
       8      7,184.34          0.16          0.19              0.27           0.71
       9      7,599.42          0.15          0.19              0.26           0.74
       10     7,132.36          0.15          0.19              0.26           0.74
       11     7,459.15          0.15          0.19              0.25           0.74
       12     7,320.64          0.15          0.19              0.26           0.74
춘천시    7      1,796.43          0.13          0.25              0.26           0.95
       8      1,860.69          0.14          0.25              0.27           0.96
       9      1,954.77          0.16          0.24              0.26           0.92
       10     1,889.35          0.14          0.25              0.25           0.99
       11     1,861.33          0.14          0.25              0.24           0.95
       12     1,774.36          0.14          0.25              0.25           0.91

- 강남구는 **주말/평일 비율이 1보다 크게 낮은 평일(직장인) 중심 상권**, 춘천시는 주말에도 유동인구가 유지되고 **고령층 비중이 더 높다**. 같은 날씨라도 두 지역 상권의 반응이 다를 수 있는 구조적 근거로 EDA에서 활용한다.

## 7. 기상 데이터 (외부 공공데이터)

| 파일 | 출처 | 범위 | 용도 |
|---|---|---|---|
| `kma_asos_hourly_2025H2.csv` | 공공데이터포털 · 기상청_지상(종관, ASOS) 시간자료 조회서비스 | 서울(108)·춘천(101), 2025-07-01 ~ 12-31 | 일·6시간 단위 기상, 기상 유형 판정 |
| `kma_asos_daily_2015_2024.csv` | 공공데이터포털 · 기상청_지상(종관, ASOS) 일자료 조회서비스 | 같은 관측소, 2015 ~ 2024 | 평년값(기준 날씨), 극한기상 발생 빈도 |

관측소 대응: **강남구 → 서울(108, 종로구 송월동)**, 강남구 안에는 종관 관측소가 없어 서울 대표 관측소를 쓴다. **춘천시 → 춘천(101)**.

In [15]:
def find_file(name):
    for d in [DATA / 'external', DATA]:
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(name)

ASOS_HOURLY = find_file('kma_asos_hourly_2025H2.csv')
ASOS_DAILY = find_file('kma_asos_daily_2015_2024.csv')
STN_TO_REGION = {108: '강남구', 101: '춘천시'}

H_COLS = {'tm': 'datetime', 'stnId': 'stn', 'ta': 'temp', 'rn': 'rain', 'hm': 'humidity', 'ws': 'wind',
          'dsnw': 'snow_depth', 'hr3Fhsc': 'snow_new_3h'}
h_raw = pd.read_csv(ASOS_HOURLY, dtype=str, keep_default_na=False)
hourly = h_raw[list(H_COLS)].rename(columns=H_COLS)
for c in ['temp', 'rain', 'humidity', 'wind', 'snow_depth', 'snow_new_3h']:
    hourly[c] = pd.to_numeric(hourly[c].replace('', np.nan), errors='coerce')
hourly['datetime'] = pd.to_datetime(hourly['datetime'])
hourly['region'] = hourly['stn'].astype(int).map(STN_TO_REGION)
hourly = hourly.sort_values(['region', 'datetime']).reset_index(drop=True)

print(f'시간자료 {len(hourly):,}행 (기대값 184일 × 24시간 × 2 = {184 * 24 * 2:,})')
miss = hourly[['temp', 'rain', 'humidity', 'wind', 'snow_depth', 'snow_new_3h']].isna().sum()
miss.to_frame('빈 값 수').T

시간자료 8,832행 (기대값 184일 × 24시간 × 2 = 8,832)


,temp,rain,humidity,wind,snow_depth,snow_new_3h
빈 값 수,1,7584,0,0,8570,8827


**빈 값 처리 기준**
- `rain`(강수량)·`snow_depth`(적설)·`snow_new_3h`(3시간 신적설): ASOS는 **비·눈이 없을 때 값을 비워서** 제공한다 → 0으로 채운다.
- `temp`(기온) 1건: 같은 관측소의 앞뒤 시각으로 선형 보간한다.

In [16]:
for c in ['rain', 'snow_depth', 'snow_new_3h']:
    hourly[c] = hourly[c].fillna(0)
hourly['temp'] = hourly.groupby('region')['temp'].transform(lambda s: s.interpolate(limit_direction='both'))
assert hourly[['temp', 'humidity', 'wind']].notna().all().all()

### 7-1. 체감온도
기상청 체감온도 공식을 그대로 쓴다.
- **여름철(5~9월)**: 기온·습도로 습구온도(Stull 식)를 구한 뒤 체감온도 계산 → 폭염 판정에 사용
- **겨울철(10~4월)**: 기온 10℃ 이하·풍속 1.3m/s 이상에서 풍속냉각 체감온도, 그 외는 기온 → 한파 판정에 사용

In [17]:
def feels_like_summer(t, rh):
    tw = (t * np.arctan(0.151977 * np.sqrt(rh + 8.313659)) + np.arctan(t + rh) - np.arctan(rh - 1.67633)
          + 0.00391838 * rh ** 1.5 * np.arctan(0.023101 * rh) - 4.686035)
    return -0.2442 + 0.55399 * tw + 0.45535 * t - 0.0022 * tw ** 2 + 0.00278 * tw * t + 3.0

def wind_chill(t, ws):
    v = (ws * 3.6) ** 0.16
    return np.where((t <= 10) & (ws >= 1.3), 13.12 + 0.6215 * t - 11.37 * v + 0.3965 * t * v, t)

def feels_like(month, t, rh, ws):
    return np.where(pd.Series(month).between(5, 9).values, feels_like_summer(t, rh), wind_chill(t, ws))

hourly['feels_like'] = feels_like(hourly['datetime'].dt.month, hourly['temp'], hourly['humidity'], hourly['wind'])
hourly['rain_3h'] = hourly.groupby('region')['rain'].transform(lambda s: s.rolling(3, min_periods=1).sum())
hourly['date'] = hourly['datetime'].dt.normalize()
hourly['time_slot'] = pd.cut(hourly['datetime'].dt.hour, [-1, 5, 11, 17, 23], labels=TIME_SLOTS)

### 7-2. 일 단위 기상과 기상 유형

**신적설(새로 쌓인 눈)** = 그날 최대 적설 − 전날 마지막 시각 적설 (0 미만은 0). 서울은 3시간 신적설 관측값도 있어 둘 중 큰 값을 쓴다.
적설계 값은 눈이 녹거나 다져지면서 0.1~0.3cm씩 오르내리므로, **0.5cm 이상 늘어난 날만 '눈 온 날'**로 본다.

| 플래그 | 기준 | 근거 |
|---|---|---|
| `heat_day` | 일최고 체감온도 ≥ 33℃ | 폭염주의보 기준 (2일 지속 조건은 미반영) |
| `tropical_night` | 일최저기온 ≥ 25℃ | 열대야 |
| `rain_day` | 일강수량 ≥ 1mm | 강수일 |
| `heavy_rain_day` | 일강수량 ≥ 30mm | 강한 비 (분석 정의) |
| `rain_warning` | 3시간 누적 강수 ≥ 60mm | 호우주의보 3시간 기준 |
| `cold_day` | 일최저기온 ≤ -12℃ | 한파주의보 기준값 (전날 대비 조건 미반영) |
| `freeze_day` | 일최고기온 < 0℃ | 결빙일 |
| `snow_day` | 신적설 ≥ 0.5cm | 눈이 쌓인 날 (적설계 흔들림 제외) |
| `heavy_snow_day` | 신적설 ≥ 5cm | 대설주의보 기준 |

`weather_type`은 하루를 하나의 유형으로 보는 라벨이다. 겹치면 **눈 > 강한 비 > 폭염 > 한파 > 비 > 평상** 순으로 정한다 (드문 사건 우선).

In [18]:
daily_w = (hourly.groupby(['date', 'region'])
                 .agg(temp_mean=('temp', 'mean'), temp_max=('temp', 'max'), temp_min=('temp', 'min'),
                      feels_like_max=('feels_like', 'max'), feels_like_min=('feels_like', 'min'),
                      humidity_mean=('humidity', 'mean'), humidity_min=('humidity', 'min'),
                      wind_mean=('wind', 'mean'), wind_max=('wind', 'max'),
                      rain_sum=('rain', 'sum'), rain_max_1h=('rain', 'max'), rain_max_3h=('rain_3h', 'max'),
                      rain_hours=('rain', lambda s: int((s > 0).sum())),
                      snow_depth_max=('snow_depth', 'max'), snow_depth_last=('snow_depth', 'last'),
                      snow_new_3h_sum=('snow_new_3h', 'sum'))
                 .reset_index())
prev_last = daily_w.groupby('region')['snow_depth_last'].shift(1).fillna(0)
daily_w['snow_new'] = np.maximum((daily_w['snow_depth_max'] - prev_last).clip(lower=0), daily_w['snow_new_3h_sum'])
daily_w = daily_w.drop(columns=['snow_depth_last', 'snow_new_3h_sum'])

daily_w['heat_day'] = daily_w['feels_like_max'] >= 33
daily_w['tropical_night'] = daily_w['temp_min'] >= 25
daily_w['rain_day'] = daily_w['rain_sum'] >= 1
daily_w['heavy_rain_day'] = daily_w['rain_sum'] >= 30
daily_w['rain_warning'] = daily_w['rain_max_3h'] >= 60
daily_w['cold_day'] = daily_w['temp_min'] <= -12
daily_w['freeze_day'] = daily_w['temp_max'] < 0
daily_w['snow_day'] = daily_w['snow_new'] >= 0.5
daily_w['heavy_snow_day'] = daily_w['snow_new'] >= 5

daily_w['heat_bin'] = pd.cut(daily_w['feels_like_max'], [-np.inf, 30, 33, 35, np.inf],
                             labels=['~30', '30~33', '33~35', '35~'], right=False)
daily_w['rain_bin'] = pd.cut(daily_w['rain_sum'], [-np.inf, 0, 5, 30, 80, np.inf],
                             labels=['없음', '약(0~5)', '보통(5~30)', '강(30~80)', '극한(80~)'])
WEATHER_TYPES = ['눈', '강한 비', '폭염', '한파', '비', '평상']
daily_w['weather_type'] = pd.Categorical(np.select(
    [daily_w['snow_day'], daily_w['heavy_rain_day'], daily_w['heat_day'], daily_w['cold_day'], daily_w['rain_day']],
    WEATHER_TYPES[:-1], '평상'), categories=WEATHER_TYPES)

flags = ['heat_day', 'tropical_night', 'rain_day', 'heavy_rain_day', 'rain_warning',
         'cold_day', 'freeze_day', 'snow_day', 'heavy_snow_day']
print('2025년 7~12월 기상 플래그별 일수')
display(daily_w.groupby('region')[flags].sum().T)
print('weather_type 일수 (월별)')
pd.crosstab([daily_w['region'], daily_w['date'].dt.month.rename('월')], daily_w['weather_type'])

2025년 7~12월 기상 플래그별 일수


region,강남구,춘천시
heat_day,25,19
tropical_night,45,5
rain_day,47,46
heavy_rain_day,13,17
rain_warning,0,1
cold_day,0,3
freeze_day,4,4
snow_day,2,3
heavy_snow_day,1,1


weather_type 일수 (월별)


weather_type  눈  강한 비  폭염  한파  비  평상
region 월                            
강남구    7      0     3  15   0  4   9
       8      0     2  10   0  5  14
       9      0     7   0   0  4  19
       10     0     1   0   0  8  22
       11     0     0   0   0  3  27
       12     2     0   0   0  3  26
춘천시    7      0     3  12   0  6  10
       8      0     4   7   0  4  16
       9      0     7   0   0  5  18
       10     0     3   0   0  6  22
       11     0     0   0   0  3  27
       12     3     0   0   2  2  24

- 폭염·강한 비는 7~9월, 눈은 12월에만 나타난다. **같은 6개월 안에 여름 극한기상과 겨울 극한기상이 모두 있어** 가설 3(기상 유형별 업종 피해 차이)을 비교할 수 있다.
- 다만 눈·한파 일수는 매우 적어(지역별 수 일) 통계적 비교보다는 **사건(이벤트) 단위 비교**가 현실적이다. 이 점은 기술통계·EDA에서 다시 확인한다.

In [19]:
slot_w = (hourly.groupby(['date', 'time_slot', 'region'], observed=True)
                .agg(slot_temp_mean=('temp', 'mean'), slot_temp_max=('temp', 'max'), slot_temp_min=('temp', 'min'),
                     slot_feels_like_max=('feels_like', 'max'), slot_feels_like_min=('feels_like', 'min'),
                     slot_humidity_mean=('humidity', 'mean'), slot_wind_mean=('wind', 'mean'),
                     slot_rain_sum=('rain', 'sum'), slot_rain_max_1h=('rain', 'max'),
                     slot_rain_hours=('rain', lambda s: int((s > 0).sum())), slot_snow_depth_max=('snow_depth', 'max'),
                     slot_n_hours=('temp', 'size'))
                .reset_index())
assert len(daily_w) == 184 * 2 and len(slot_w) == 184 * 4 * 2 and (slot_w['slot_n_hours'] == 6).all()
print('일 단위', daily_w.shape, '| 6시간 단위', slot_w.shape)

일 단위 (368, 29) | 6시간 단위 (1472, 15)


### 7-3. 평년값과 극한기상 빈도 (2015~2024 일자료)

- **평년값**: 같은 지역·같은 월일 **±7일(15일 × 10년 = 150개 관측)**의 평균과 발생 확률. 피해 예측에서 '평소 날씨'(기준 시나리오)로 쓴다.
- **연도별 극한기상 일수**: 지역이 앞으로 해마다 몇 번 그 기상을 겪는지(노출 빈도) — 연간 피해 규모 환산에 쓴다.
- 일자료에는 시간별 습도가 없어 체감온도는 **일최고기온 + 일최저습도**로 근사한다.

In [20]:
D_COLS = {'tm': 'date', 'stnId': 'stn', 'avgTa': 'temp_mean', 'minTa': 'temp_min', 'maxTa': 'temp_max',
          'sumRn': 'rain_sum', 'hr1MaxRn': 'rain_max_1h', 'avgRhm': 'humidity_mean', 'minRhm': 'humidity_min',
          'avgWs': 'wind_mean', 'ddMefs': 'snow_new', 'ddMes': 'snow_depth_max'}
d_raw = pd.read_csv(ASOS_DAILY, dtype=str, keep_default_na=False)
clim = d_raw[list(D_COLS)].rename(columns=D_COLS)
for c in list(D_COLS.values())[2:]:
    clim[c] = pd.to_numeric(clim[c].replace('', np.nan), errors='coerce')
for c in ['rain_sum', 'rain_max_1h', 'snow_new', 'snow_depth_max']:   # 비·눈 없는 날은 공란
    clim[c] = clim[c].fillna(0)
clim['date'] = pd.to_datetime(clim['date'])
clim['region'] = clim['stn'].astype(int).map(STN_TO_REGION)
clim = clim.sort_values(['region', 'date']).reset_index(drop=True)

# 신적설: 일 최심신적설(ddMefs)과 일 최심적설(ddMes)의 전날 대비 증가분 중 큰 값
depth_gain = (clim['snow_depth_max'] - clim.groupby('region')['snow_depth_max'].shift(1)).clip(lower=0).fillna(0)
clim['snow_new'] = np.maximum(clim['snow_new'], depth_gain)
# 적설 관측이 한 번도 없는 해는 '눈이 안 온 해'가 아니라 '관측이 없는 해'로 보고 결측 처리
clim['year'] = clim['date'].dt.year
snow_obs = clim.groupby(['region', 'year'])['snow_depth_max'].transform(lambda s: (s > 0).any())
clim.loc[~snow_obs, ['snow_new', 'snow_depth_max']] = np.nan
print('적설 관측이 있는 해:', clim[snow_obs].groupby('region')['year'].unique().apply(lambda v: [int(x) for x in v]).to_dict())
clim['feels_like_max'] = feels_like(clim['date'].dt.month, clim['temp_max'], clim['humidity_min'], clim['wind_mean'])
print(f"일자료 {len(clim):,}행 | 지역별 일수 {clim.groupby('region').size().to_dict()} | "
      f"기온 결측 {clim['temp_max'].isna().sum()}일")

# 평년값: ±7일 이동창 (2/29 제외, 연말·연초 순환)
c2 = clim[~((clim['date'].dt.month == 2) & (clim['date'].dt.day == 29))].copy()
cal365 = pd.date_range('2025-01-01', '2025-12-31')
doy = {d.strftime('%m-%d'): i for i, d in enumerate(cal365)}
c2['doy'] = c2['date'].dt.strftime('%m-%d').map(doy)
rows = []
for region, g in c2.groupby('region'):
    for i, day in enumerate(cal365):
        dist = (g['doy'] - i).abs()
        w = g[np.minimum(dist, 365 - dist) <= 7]
        rows.append({'region': region, 'month_day': day.strftime('%m-%d'),
                     'normal_temp_mean': w['temp_mean'].mean(), 'normal_temp_max': w['temp_max'].mean(),
                     'normal_temp_min': w['temp_min'].mean(), 'normal_feels_like_max': w['feels_like_max'].mean(),
                     'normal_humidity_mean': w['humidity_mean'].mean(), 'normal_wind_mean': w['wind_mean'].mean(),
                     'normal_rain_sum': w['rain_sum'].mean(), 'normal_snow_new': w['snow_new'].mean(),
                     'prob_rain_day': (w['rain_sum'] >= 1).mean(), 'prob_heavy_rain_day': (w['rain_sum'] >= 30).mean(),
                     'prob_heat_day': (w['temp_max'] >= 33).mean(), 'prob_cold_day': (w['temp_min'] <= -12).mean(),
                     'prob_snow_day': (w['snow_new'].dropna() >= 0.5).mean(), 'n_obs': len(w)})
weather_normal = pd.DataFrame(rows)

yr = clim
weather_climate = yr.groupby(['region', 'year']).agg(
    heat_days=('temp_max', lambda s: int((s >= 33).sum())),
    tropical_nights=('temp_min', lambda s: int((s >= 25).sum())),
    rain_days=('rain_sum', lambda s: int((s >= 1).sum())),
    heavy_rain_days=('rain_sum', lambda s: int((s >= 30).sum())),
    extreme_rain_days=('rain_sum', lambda s: int((s >= 80).sum())),
    cold_days=('temp_min', lambda s: int((s <= -12).sum())),
    snow_days=('snow_new', lambda s: (s >= 0.5).sum() if s.notna().any() else np.nan),
    heavy_snow_days=('snow_new', lambda s: (s >= 5).sum() if s.notna().any() else np.nan),
    n_days=('date', 'size')).reset_index()

print('연평균 극한기상 일수 (2015~2024, 기온 기준은 실제 기온 / 눈은 적설 관측이 있는 해만 평균)')
weather_climate.drop(columns=['year', 'n_days']).groupby('region').mean().round(1)

적설 관측이 있는 해: {'강남구': [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024], '춘천시': [2015, 2016, 2023]}
일자료 7,305행 | 지역별 일수 {'강남구': 3653, '춘천시': 3652} | 기온 결측 1일


연평균 극한기상 일수 (2015~2024, 기온 기준은 실제 기온 / 눈은 적설 관측이 있는 해만 평균)


,heat_days,tropical_nights,rain_days,heavy_rain_days,extreme_rain_days,cold_days,snow_days,heavy_snow_days
region,,,,,,,,
강남구,17.90,21.40,77.50,13.00,2.50,4.40,7.60,1.30
춘천시,19.00,8.80,73.40,12.00,1.80,16.80,3.70,0.30


- **춘천 관측소는 2017~2022·2024년 일자료에 적설 기록이 없다** (2025년 시간자료에는 적설계 값이 있음). 이 해들은 '눈이 안 온 해'가 아니라 관측 공백으로 보고 결측 처리했다. 따라서 춘천의 연평균 눈 일수는 2015·2016·2023년 3개 해로만 계산된 값이라 **신뢰도가 낮다.**
- 춘천은 한파일(최저 -12℃ 이하)이 강남의 약 4배로, 겨울 기상 노출은 눈보다 **한파** 쪽이 뚜렷하다.

In [21]:
# 2025년 하반기는 평년과 비교해 어땠나
cmp = daily_w.assign(month=daily_w['date'].dt.month, month_day=daily_w['date'].dt.strftime('%m-%d')).merge(
    weather_normal, on=['region', 'month_day'])
cmp = cmp.groupby(['region', 'month']).agg(
    최고기온=('temp_max', 'mean'), 평년_최고기온=('normal_temp_max', 'mean'),
    강수합=('rain_sum', 'sum'), 평년_강수합=('normal_rain_sum', 'sum'))
cmp.round(1)

최고기온  평년_최고기온    강수합  평년_강수합
region month                              
강남구    7     32.40    30.40 293.90  329.10
       8     31.80    31.00 290.30  255.60
       9     27.20    27.20 370.70  123.90
       10    20.30    20.90 186.20   58.70
       11    14.00    12.80  20.90   69.40
       12     5.10     4.60  41.20   30.20
춘천시    7     31.90    30.40 286.00  319.60
       8     31.60    30.60 252.90  253.60
       9     26.60    26.30 381.40  131.70
       10    19.90    19.80 157.20   58.90
       11    13.60    12.10  19.80   60.20
       12     3.90     3.80  30.10   29.50

## 8. 소상공인 상가(상권)정보 — 업종별 점포 수

| 항목 | 내용 |
|---|---|
| 출처 | 공공데이터포털 · 소상공인시장진흥공단_상가(상권)정보 (2026년 6월 기준, 파일데이터) |
| 사용 범위 | 서울·강원 파일 중 시군구코드 **11680(강남구)·51110(춘천시)** — 행정안전부 코드라 SKT 코드(11230·32010)와 체계가 다르다 |
| 용도 | 업종별 **점포 수** → 점포당 매출·점포당 피해 (가설 1·2), **체인 비중**(지점명 유무) → 대응 여력 |

- 상가정보 업종(대·중·소분류)을 카드 업종대분류에 맞춘 매핑표를 만들었다 (`references/store_industry_map.csv`). 소분류가 채워진 행이 중분류 매핑보다 우선한다 (예: 기타 간이 중 빵/도넛 → 카페·디저트).
- **기준 시점 차이**: 상가정보는 2026년 6월, 카드는 2025년 하반기다. 점포 수는 업종 규모를 나타내는 지표로만 쓰고 개별 점포와 매출을 연결하지 않는다.

In [22]:
STORE_DIR = DATA / '소상공인시장진흥공단_상가(상권)정보_20260630'
STORE_SGG = {'11680': '강남구', '51110': '춘천시'}
USECOLS = ['상가업소번호', '지점명', '상권업종대분류명', '상권업종중분류명', '상권업종소분류명',
           '시군구코드', '행정동코드', '행정동명', '경도', '위도']
frames = []
for sido in ['서울', '강원']:
    f = next(STORE_DIR.glob(f'*_{sido}_*.csv'))
    s = pd.read_csv(f, dtype=str, usecols=USECOLS, encoding='utf-8')
    frames.append(s[s['시군구코드'].isin(STORE_SGG)])
store_raw = pd.concat(frames, ignore_index=True).apply(lambda s: s.str.strip())

smap = pd.read_csv(REF / 'store_industry_map.csv', dtype=str, keep_default_na=False)
by_sub = smap[smap['sub'] != ''].set_index(['mid', 'sub'])
by_mid = smap[smap['sub'] == ''].set_index('mid')
key = pd.MultiIndex.from_frame(store_raw[['상권업종중분류명', '상권업종소분류명']])
grp = pd.Series(by_sub['industry_group'].reindex(key).values, index=store_raw.index)
inc = pd.Series(by_sub['analysis_include'].reindex(key).values, index=store_raw.index)
grp = grp.fillna(store_raw['상권업종중분류명'].map(by_mid['industry_group']))
inc = inc.fillna(store_raw['상권업종중분류명'].map(by_mid['analysis_include']))
assert grp.notna().all(), f"매핑표에 없는 중분류: {set(store_raw.loc[grp.isna(), '상권업종중분류명'])}"

stores = pd.DataFrame({
    'store_id': store_raw['상가업소번호'], 'region': store_raw['시군구코드'].map(STORE_SGG),
    'dong_code': store_raw['행정동코드'], 'dong_name': store_raw['행정동명'],
    'category_large': store_raw['상권업종대분류명'], 'category_mid': store_raw['상권업종중분류명'],
    'category_small': store_raw['상권업종소분류명'], 'industry_group': grp, 'analysis_include': inc.eq('Y'),
    'is_chain': store_raw['지점명'].fillna('').ne(''),   # 지점명이 있으면 체인·프랜차이즈 지점으로 간주 (근사)
    'lon': store_raw['경도'].astype(float), 'lat': store_raw['위도'].astype(float)})
assert not stores['store_id'].duplicated().any()
print(f"점포 {len(stores):,}곳 | 분석 대상 업종 {stores['analysis_include'].sum():,}곳")
stores.groupby('region')['analysis_include'].agg(['size', 'sum']).rename(columns={'size': '전체', 'sum': '분석 대상'})

점포 84,253곳 | 분석 대상 업종 51,306곳


,전체,분석 대상
region,,
강남구,66269,36380
춘천시,17984,14926


In [23]:
target_st = stores[stores['analysis_include']]
store_group = (target_st.groupby(['region', 'industry_group'])
                        .agg(n_stores=('store_id', 'size'), chain_share=('is_chain', 'mean')).reset_index())
store_dong = (target_st.groupby(['region', 'dong_code', 'dong_name', 'industry_group'])
                       .size().rename('n_stores').reset_index())

# 카드 업종대분류와 상가정보 업종대분류가 모두 대응되는지
card_groups = set(groups)
print('카드에만 있는 업종대분류:', card_groups - set(store_group['industry_group']) or '없음')
print('상가정보에만 있는 업종대분류:', set(store_group['industry_group']) - card_groups or '없음')
view = store_group.pivot(index='industry_group', columns='region', values=['n_stores', 'chain_share'])
view.sort_values(('n_stores', '강남구'), ascending=False).round(2)

카드에만 있는 업종대분류: 없음
상가정보에만 있는 업종대분류: 없음


n_stores          chain_share     
region              강남구      춘천시         강남구  춘천시
industry_group                                   
음식점            7,256.00 3,333.00        0.20 0.11
교육             5,164.00 1,218.00        0.04 0.04
의료·보건          3,825.00   579.00        0.01 0.02
미용·개인서비스       3,629.00 1,537.00        0.09 0.06
의류·잡화          3,499.00 1,033.00        0.14 0.13
기타소매           3,122.00 1,362.00        0.12 0.10
카페·디저트         2,671.00 1,128.00        0.29 0.15
간편식            1,474.00   836.00        0.35 0.29
여가·오락          1,384.00   688.00        0.11 0.08
주점·유흥          1,227.00   610.00        0.08 0.07
식료품소매            863.00   490.00        0.13 0.06
편의점              803.00   373.00        0.46 0.38
숙박               537.00   818.00        0.02 0.00
교통·자동차           474.00   514.00        0.13 0.05
대형유통             452.00   407.00        0.17 0.10

- 카드 업종대분류 15개가 상가정보에서 모두 대응된다.
- 체인 비중은 '지점명이 있는 점포' 비율로 근사한 값이다. 독립 점포일수록 기상 충격을 흡수할 여력이 적다는 가정으로 이후 단계에서 쓴다.

## 9. 분석 테이블 생성 (카드 × 기상 결합)

5절의 마트에 기상(7절)과 평년값을 붙여 이후 단계의 입력으로 쓴다.
- 일 단위 마트 ← `weather_daily` (+ 평년값, 평년 대비 편차 `anom_*`)
- 시간대 마트 ← `weather_slot` (해당 6시간 기상) + 그날의 일 기상
- 평년 대비 편차: `anom_temp_max`, `anom_feels_like_max`, `anom_rain_sum` = 실제 − 같은 지역·월일 평년값

In [24]:
DAY_W = ['temp_mean', 'temp_max', 'temp_min', 'feels_like_max', 'feels_like_min', 'humidity_mean', 'wind_mean',
         'rain_sum', 'rain_max_1h', 'rain_max_3h', 'rain_hours', 'snow_depth_max', 'snow_new'] + flags + \
        ['heat_bin', 'rain_bin', 'weather_type']
NORMAL_W = ['normal_temp_max', 'normal_temp_min', 'normal_feels_like_max', 'normal_humidity_mean',
            'normal_wind_mean', 'normal_rain_sum', 'prob_rain_day', 'prob_heat_day', 'prob_snow_day']

weather_day_full = daily_w[['date', 'region'] + DAY_W].assign(month_day=daily_w['date'].dt.strftime('%m-%d'))
weather_day_full = weather_day_full.merge(weather_normal[['region', 'month_day'] + NORMAL_W], on=['region', 'month_day'])
weather_day_full['anom_temp_max'] = weather_day_full['temp_max'] - weather_day_full['normal_temp_max']
weather_day_full['anom_feels_like_max'] = weather_day_full['feels_like_max'] - weather_day_full['normal_feels_like_max']
weather_day_full['anom_rain_sum'] = weather_day_full['rain_sum'] - weather_day_full['normal_rain_sum']

def with_day_weather(mart):
    return mart.merge(weather_day_full, on=['date', 'region'], how='left', validate='many_to_one')

analysis = {
    'analysis_daily_group': with_day_weather(marts['mart_daily_group']),
    'analysis_daily_industry': with_day_weather(marts['mart_daily_industry']),
    'analysis_origin_group': with_day_weather(marts['mart_origin_group']),
    'analysis_daily_demo': with_day_weather(marts['mart_daily_demo']),
    'analysis_slot_group': with_day_weather(marts['mart_slot_group']).merge(
        slot_w.drop(columns='slot_n_hours'), on=['date', 'time_slot', 'region'], how='left', validate='many_to_one'),
}
for name, a in analysis.items():
    assert len(a) == len(marts[name.replace('analysis', 'mart')]), name
    assert a['temp_max'].notna().all() and a['normal_temp_max'].notna().all(), name
pd.DataFrame({n: {'행': len(a), '열': a.shape[1]} for n, a in analysis.items()}).T

,행,열
analysis_daily_group,5520,51
analysis_daily_industry,26128,52
analysis_origin_group,16560,52
analysis_daily_demo,6624,52
analysis_slot_group,22080,63


## 10. 검증 요약·저장

In [25]:
checks = {
    '데이터1 행 수·금액 보존': len(card1) == len(raw1) and card1['amount'].sum() == raw1['TS_AT'].astype('int64').sum(),
    '데이터2 행 수·금액 보존': len(card2) == len(raw2) and card2['amount'].sum() == raw2['TS_AT'].astype('int64').sum(),
    '업종 매핑 누락 없음': card1['industry_group'].notna().all() and card2['industry_group'].notna().all(),
    '개인 결제는 성별·연령 존재': card1.loc[card1['customer_type'] == '개인', ['sex', 'age']].notna().all().all(),
    '데이터1(개인)↔데이터2 지역 합계 0.01% 이내': bool(((reg['card2'] / reg['card1_개인'] - 1).abs() < 1e-4).all()),
    '일 단위 마트 균형 (184일 × 2지역 × 업종대분류)': len(marts['mart_daily_group']) == 184 * 2 * len(groups),
    '일 단위 마트 금액 보존': marts['mart_daily_group']['amount'].sum() == personal['amount'].sum(),
    '출신 마트 금액 보존': marts['mart_origin_group']['amount'].sum() == target2['amount'].sum(),
    '유동인구 격자×월 키 중복 없음': not flow.duplicated(['month', 'grid_id']).any(),
    '유동인구 6개월 × 2지역': len(flow_region_month) == 12,
    '기상 시간자료 184일 × 24시간 × 2지역': len(hourly) == 184 * 24 * 2,
    '평년값 365일 × 2지역': len(weather_normal) == 365 * 2,
    '상가정보 카드 업종대분류 전부 대응': card_groups <= set(store_group['industry_group']),
    '분석 테이블 기상 결합 누락 없음': all(a['temp_max'].notna().all() for a in analysis.values()),
}
for k, ok in checks.items():
    print(f"[{'OK' if ok else '실패'}] {k}")
assert all(checks.values())

[OK] 데이터1 행 수·금액 보존
[OK] 데이터2 행 수·금액 보존
[OK] 업종 매핑 누락 없음
[OK] 개인 결제는 성별·연령 존재
[OK] 데이터1(개인)↔데이터2 지역 합계 0.01% 이내
[OK] 일 단위 마트 균형 (184일 × 2지역 × 업종대분류)
[OK] 일 단위 마트 금액 보존
[OK] 출신 마트 금액 보존
[OK] 유동인구 격자×월 키 중복 없음
[OK] 유동인구 6개월 × 2지역
[OK] 기상 시간자료 184일 × 24시간 × 2지역
[OK] 평년값 365일 × 2지역
[OK] 상가정보 카드 업종대분류 전부 대응
[OK] 분석 테이블 기상 결합 누락 없음


In [26]:
outputs = {'card1_clean': card1, 'card2_clean': card2, **marts,
           'flow_grid_month': flow, 'flow_region_month': flow_region_month,
           'flow_dong_profile': flow_dong, 'flow_grid_profile': flow_grid,
           'weather_hourly': hourly, 'weather_daily': daily_w, 'weather_slot': slot_w,
           'weather_normal': weather_normal, 'weather_climate': weather_climate,
           'store_clean': stores, 'store_group_summary': store_group, 'store_dong_summary': store_dong,
           **analysis}
for name, frame in outputs.items():
    frame.to_parquet(OUT / f'{name}.parquet', index=False)
pd.DataFrame({'행': {n: len(f) for n, f in outputs.items()}, '열': {n: f.shape[1] for n, f in outputs.items()}})

,행,열
card1_clean,1044710,18
card2_clean,1806800,18
mart_daily_group,5520,13
mart_daily_industry,26128,14
mart_slot_group,22080,14
mart_origin_group,16560,14
mart_daily_demo,6624,14
flow_grid_month,720677,73
flow_region_month,12,22
flow_dong_profile,47,22


## 정리

| 항목 | 결과 |
|---|---|
| 카드 데이터 품질 | 결측·중복 없음. 결제 5건 미만 칸은 원본에 없음 → 0 채움 = '소액 미집계' 포함 |
| 분석 범위 | 개인 결제 × 분석 포함 업종(71개 업종, 17개 대분류 중 15개). 세금·보험·미분류·법인은 제외 |
| 데이터 간 관계 | 데이터1 개인 = 데이터2 (날짜×지역×업종 단위로 일치) → 시간대 분석과 고객출신 분석을 같은 총액으로 해석 |
| 유동인구 | 12월 시간대·요일 파일의 완전 중복 제거, 인접 시군구 격자 제외, 세 파일 척도 일치 확인 |
| 기상 | 체감온도(기상청 공식), 신적설, 기상 플래그·유형(눈·강한 비·폭염·한파·비·평상), 평년값(±7일 × 10년), 연도별 극한기상 일수 |
| 상가정보 | 강남·춘천 점포의 업종대분류·체인 여부·행정동 → 업종별 점포 수 (점포당 매출·피해의 분모) |
| 분석 테이블 | `analysis_*` 5종 = 카드 마트 + 일·시간대 기상 + 평년 대비 편차 |
| 다음 단계 | `02_descriptive_stats.ipynb` (기술통계) |